# SetUp

In [73]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.85'

In [74]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("../../")
    
print(f"Directorio de trabajo actual: {os.getcwd()}")

Directorio de trabajo actual: /home/alanh/Dev/owns/thesis


# Neat

In [75]:
import jax, jax.numpy as jnp
from tensorneat import algorithm, genome
from tensorneat.pipeline import Pipeline
from tensorneat.genome import DefaultGenome, BiasNode, DefaultMutation
from tensorneat.problem.func_fit import CustomFuncFit
from tensorneat.common import ACT, AGG, State
from tensorneat.common.functions import act_jnp
from tensorneat.pipeline import Pipeline
from tensorneat.algorithm.neat import NEAT
from tensorneat.algorithm.hyperneat import HyperNEAT, FullSubstrate
import h5py
import numpy as np
from tqdm.auto import tqdm

## Utils


In [76]:
def check_vram_usage():
    device = jax.devices("gpu")[0]  # Accedemos a tu RTX 4060
    stats = device.memory_stats()
    
    used_gb = stats['bytes_in_use'] / 1e9
    limit_gb = stats['bytes_limit'] / 1e9
    peak_gb = stats['peak_bytes_in_use'] / 1e9
    reserved_gb = stats.get('pool_bytes', 0) / 1e9 # Memoria que JAX ya pidió al OS
    
    print(f"📊 --- ESTADO DE VRAM (JAX) ---")
    print(f"🔹 En uso real:  {used_gb:.2f} GB  (Tensores actuales)")
    print(f"📈 Pico máximo:  {peak_gb:.2f} GB")
    print(f"📦 Reservado:    {reserved_gb:.2f} GB  (Lo que ves en nvidia-smi)")
    print(f"🚫 Límite JAX:   {limit_gb:.2f} GB")
    
    return used_gb, peak_gb

## Constants

In [ ]:
DATA_PATH = "crystals/"
CIF_PATH = DATA_PATH + "cif/"
METADATA_PATH = DATA_PATH + "_metadata.json"
EMBEDDINGS_PATH = DATA_PATH + "_embeddings.json"
FINAL_DATA_PATH = DATA_PATH + "_data.json"
DATASET_PATH = DATA_PATH + "dataset_fase1a.h5" 

# Neat
POPSIZE = 72
SPECIES_SIZE = 8
SURVIVAL_THRESHOLD = 0.1
MAX_NODES = 200
MAX_CONNS = 2200
CONN_ADD_PROB = 0.6
CONN_DELETE_PROB = 0.3
NODE_ADD_PROB = 0.1
NODE_DELETE_PROB = 0.05

# Structure
MAX_ELEMENTS = 2       # Tipos únicos de elemento por estructura
ELEM_FEATURES = 9      # Features por elemento
MAX_ATOMS = 4         # Límite de átomos en output
CRYSTAL_EMBED = 64     # CHGNet embedding
CRYSTAL_PROPS = 3      # bandgap, e_hull, density
LATTICE_PARAMS = 6     # a, b, c, alpha, beta, gamma
STRUCTURE_DIM = MAX_ATOMS * 4 # (Z, x, y, z)

# Training Loop
BATCH_SIZE = 4
INPUT_DIM  = (MAX_ELEMENTS * ELEM_FEATURES) + CRYSTAL_EMBED + CRYSTAL_PROPS
OUTPUT_DIM = LATTICE_PARAMS + STRUCTURE_DIM
N_GENERATIONS = 500
GRAD_STEPS_PER_GEN = 5
GRAD_LR = 0.005
LOG_EVERY = 1

SEED = 42

## Data

### Utils

In [78]:
def load_crystal_dataset(h5_path):
    with h5py.File(h5_path, 'r') as hf:
        np_inputs  = hf['inputs'][:]
        np_targets = hf['targets'][:]
        np_ids     = hf['material_ids'][:].astype(str)
    
    mb = (np_inputs.nbytes + np_targets.nbytes) / 1e6
    print(f"Dataset cargado en RAM: {mb:.1f} MB")
    print(f"Muestras: {len(np_inputs)}")
    return np_inputs, np_targets, np_ids



# 2. El "DataLoader" nativo para JAX
class JAXBatchLoader:
    def __init__(self, x, y, batch_size, drop_last=True):
        self.x = x
        self.y = y
        self.batch_size = batch_size
        self.num_samples = len(x)
        # drop_last=True es VITAL para JAX. Si un batch es más pequeño (ej. sobran 10 cristales),
        # jax.jit recompilará toda la red. Mejor descartar el remanente.
        self.drop_last = drop_last 

    def __iter__(self):
        # Mezclamos los índices en cada "epoch" (vital para que la red no memorice el orden)
        indices = np.random.permutation(self.num_samples)
        
        for i in range(0, self.num_samples, self.batch_size):
            if self.drop_last and i + self.batch_size > self.num_samples:
                break
                
            batch_idx = indices[i:i+self.batch_size]
            
            # 🚀 AQUÍ OCURRE LA TRANSFERENCIA RAM -> GPU
            # Convertimos el pedacito de Numpy a JAX array justo cuando se necesita
            yield jnp.array(self.x[batch_idx]), jnp.array(self.y[batch_idx])
            
    def __len__(self):
        if self.drop_last:
            return self.num_samples // self.batch_size
        return (self.num_samples + self.batch_size - 1) // self.batch_size

### Load

In [79]:
X_train, Y_train, material_ids = load_crystal_dataset(DATASET_PATH)
train_loader = JAXBatchLoader(X_train, Y_train, batch_size=BATCH_SIZE)    

INPUT_DIM  = X_train.shape[1]
OUTPUT_DIM = Y_train.shape[1]

print(f"INPUT_DIM:  {INPUT_DIM}")
print(f"OUTPUT_DIM: {OUTPUT_DIM}")
print(f"Cristales:  {len(material_ids)}")

Dataset cargado en RAM: 0.0 MB
Muestras: 16
INPUT_DIM:  85
OUTPUT_DIM: 22
Cristales:  16


## Algorithm

In [80]:
neat = algorithm.NEAT(
    pop_size=POPSIZE,
    species_size=SPECIES_SIZE,
    survival_threshold=SURVIVAL_THRESHOLD,
    genome=genome.DefaultGenome(
        num_inputs=INPUT_DIM,
        num_outputs=OUTPUT_DIM,
        max_nodes=MAX_NODES,
        max_conns=MAX_CONNS,      
        init_hidden_layers=(),
        output_transform=act_jnp.identity_,
        mutation=DefaultMutation(
            conn_add=CONN_ADD_PROB,         # correlation explorer
            conn_delete=CONN_DELETE_PROB,   # synaptic pruning
            node_add=NODE_ADD_PROB,         # creator of depth
            node_delete=NODE_DELETE_PROB,   # destructor of connections
        ),
    ),
)

## Train

### Loss Function

In [81]:
def crystal_loss_fn(preds, y_target):
    """
    Loss pura compatible con JAX.
    
    preds:    (batch, OUTPUT_DIM)
    y_target: (batch, OUTPUT_DIM)
    returns:  scalar
    """
    # Separar lattice y átomos
    pred_lattice   = preds[:, :LATTICE_PARAMS]
    target_lattice = y_target[:, :LATTICE_PARAMS]

    pred_atoms   = preds[:, LATTICE_PARAMS:].reshape(-1, MAX_ATOMS, 4)
    target_atoms = y_target[:, LATTICE_PARAMS:].reshape(-1, MAX_ATOMS, 4)

    # Máscara: átomo existe si Z normalizado > 0
    mask = (target_atoms[:, :, 0] > 0).astype(jnp.float32)

    # Loss 1: parámetros de red (prioridad máxima)
    lat_loss = jnp.mean((pred_lattice - target_lattice) ** 2)

    # Loss 2: tipo de elemento (Z normalizado)
    z_loss = jnp.mean((pred_atoms[:, :, 0] - target_atoms[:, :, 0]) ** 2)

    # Loss 3: posiciones xyz (solo donde hay átomo real)
    pos_loss = jnp.mean(
        mask[:, :, None] * (pred_atoms[:, :, 1:] - target_atoms[:, :, 1:]) ** 2
    )

    # Loss 4: coordenadas fuera de [0, 1]
    coords = pred_atoms[:, :, 1:]
    oob = jnp.mean(
        mask[:, :, None] * (
            jnp.maximum(0.0, coords - 1.0) ** 2 +
            jnp.maximum(0.0, -coords) ** 2
        )
    )

    return 50.0 * lat_loss + 2.0 * z_loss + 10.0 * pos_loss + 5.0 * oob

In [82]:
import jax
import jax.numpy as jnp

# 1. Instanciación correcta de la clase

# Definimos nuestro "Examen Real" (los primeros 4 cristales)
y_target = Y_train[0:1]

print("="*50)
print("🧪 BATERÍA DE PRUEBAS DE LA FUNCIÓN DE PÉRDIDA")
print("="*50)

# -------------------------------------------------------------------
# PRUEBA 1: El Escenario "Fallo Total" (Ceros)
# Simula una red que murió (pesos en cero).
# -------------------------------------------------------------------
pred_zeros = jnp.zeros_like(y_target)
loss_zeros = crystal_loss_fn(pred_zeros, y_target)
print(f"❌ Prueba 1 (Todo Ceros)          | Loss: {loss_zeros:.4f}")
print("   * Esperado: Un número alto. Penaliza ignorar el lattice y los elementos.")

# -------------------------------------------------------------------
# PRUEBA 2: El Escenario "Amnesia/Colapso" (Otros cristales)
# Simula que la red predijo los cristales equivocados (ej. predice del 4 al 7)
# -------------------------------------------------------------------
# Verificamos que tengamos al menos 8 cristales para esta prueba
if len(Y_train) >= 8:
    pred_wrong = Y_train[1:2]
    loss_wrong = crystal_loss_fn(pred_wrong, y_target)
    print(f"⚠️ Prueba 2 (Cristales Equivocados)| Loss: {loss_wrong:.4f}")
    print("   * Esperado: Medio/Alto. La red formó cristales válidos, pero no los que pedía el Input.")
else:
    print("⚠️ Prueba 2 saltada (No hay suficientes cristales en Y_train)")

# -------------------------------------------------------------------
# PRUEBA 3: El Escenario "Casi Perfecto" (1% de ruido aleatorio)
# Simula una red al final del entrenamiento tratando de ajustar decimales.
# -------------------------------------------------------------------
# Creamos ruido aleatorio uniforme entre 0.99 y 1.01 (± 1%)
key = jax.random.PRNGKey(42)
noise = jax.random.uniform(key, shape=y_target.shape, minval=0.99, maxval=1.01)

# Aplicamos el ruido al target (1% de desviación en todos los parámetros)
pred_noisy = y_target * noise
loss_noisy = crystal_loss_fn(pred_noisy, y_target)

print(f"✅ Prueba 3 (Ruido del 1%)        | Loss: {loss_noisy:.4f}")
print("   * Esperado: Un número muy pequeño, cercano a cero pero NO cero.")
print("="*50)

🧪 BATERÍA DE PRUEBAS DE LA FUNCIÓN DE PÉRDIDA
❌ Prueba 1 (Todo Ceros)          | Loss: 17.9249
   * Esperado: Un número alto. Penaliza ignorar el lattice y los elementos.
⚠️ Prueba 2 (Cristales Equivocados)| Loss: 0.3001
   * Esperado: Medio/Alto. La red formó cristales válidos, pero no los que pedía el Input.
✅ Prueba 3 (Ruido del 1%)        | Loss: 0.0001
   * Esperado: Un número muy pequeño, cercano a cero pero NO cero.


### Trainin Loop

In [83]:
state = State(randkey=jax.random.key(SEED))
state = neat.setup(state)
g = neat.genome

def single_grad_step(nodes, conns, state, x_batch, y_batch):
    """
    Gradient descent para UN individuo.
    y_batch se pasa como argumento: JAX lo trata como variable, no constante.
    """
    def loss_fn(preds):
        return crystal_loss_fn(preds, y_batch)

    loss, (grads_n, grads_c) = g.grad(state, nodes, conns, x_batch, loss_fn)
    grads_n = jnp.clip(grads_n, -1.0, 1.0)
    grads_c = jnp.clip(grads_c, -1.0, 1.0)
    return nodes - GRAD_LR * grads_n, conns - GRAD_LR * grads_c, loss

# Vectorizar sobre población y compilar UNA SOLA VEZ
batch_grad_step = jax.jit(
    jax.vmap(single_grad_step, in_axes=(0, 0, None, None, None))
)

print("batch_grad_step compilado y listo.")
print(f"Población: {POPSIZE} | Batch: {BATCH_SIZE} | Grad steps/gen: {GRAD_STEPS_PER_GEN}")

batch_grad_step compilado y listo.
Población: 72 | Batch: 4 | Grad steps/gen: 5


In [ ]:
pbar = tqdm(range(N_GENERATIONS), desc="🧬 Evolución NEAT")
history = []

WINDOW_PCT = 1.0
steps_per_epoch = max(1, len(X_train) // BATCH_SIZE)
window_size = max(1, int(steps_per_epoch * WINDOW_PCT))

for generation in pbar:
    # 1. NEAT genera la población de esta generación
    pop_nodes, pop_conns = neat.ask(state)

    # 2. Batch aleatorio de cristales para esta generación
    batch_idx = np.random.choice(len(X_train), size=BATCH_SIZE, replace=False)
    x_batch = jnp.array(X_train[batch_idx])  # (BATCH_SIZE, INPUT_DIM)
    y_batch = jnp.array(Y_train[batch_idx])  # (BATCH_SIZE, OUTPUT_DIM)

    current_mids = [material_ids[i] for i in batch_idx]
    mids_str = ",".join(current_mids)

    # 3. Gradient descent sobre toda la población con este batch
    for step in range(GRAD_STEPS_PER_GEN):
        pop_nodes, pop_conns, batch_losses = batch_grad_step(
            pop_nodes, pop_conns, state, x_batch, y_batch
        )
        if step == 0:
            loss_estudio_inicio = np.nanmin(jax.device_get(batch_losses))
        if step == GRAD_STEPS_PER_GEN - 1:
            loss_estudio_final = np.nanmin(jax.device_get(batch_losses))

    # 4. Fitness = negativo del loss (NEAT maximiza)
    cpu_losses = jax.device_get(batch_losses)
    valid = np.isfinite(cpu_losses)
    cpu_losses_safe = np.where(valid, cpu_losses, 1e6)
    fitnesses = -cpu_losses_safe

    # 5. NEAT selecciona y genera siguiente generación
    state = neat.tell(state, fitnesses)

    # 6. Telemetría y Logging Dinámico
    bl = float(np.min(cpu_losses_safe[valid])) if valid.any() else float('nan')
    
    history.append({
        'gen': generation,
        'init_loss': loss_estudio_inicio,
        'best_loss': bl,
        'valid': int(valid.sum())
    })

    device = jax.devices("gpu")[0]
    stats = device.memory_stats()
    peak_gb = stats['peak_bytes_in_use'] / 1e9

    ventana = history[-window_size:]
    init_mean = np.mean([h['init_loss'] for h in ventana])
    best_mean = np.mean([h['best_loss'] for h in ventana])
    
    pbar.set_postfix({
        'IDs': mids_str[:BATCH_SIZE],
        f'InitMean({int(WINDOW_PCT*100)}%)': f"{init_mean:.4f}",
        f'BestMean({int(WINDOW_PCT*100)}%)': f"{best_mean:.4f}",
        'V': f"{valid.sum()}/{POPSIZE}",
        'VRAMPeak': f"{peak_gb:.2f}"
    })

print("\n✅ Entrenamiento finalizado.")

🧬 Evolución NEAT:   0%|          | 0/500 [00:00<?, ?it/s]

E0413 17:05:31.279640    4919 slow_operation_alarm.cc:73] 
********************************
[Compiling module jit_while for GPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
E0413 17:05:48.096187    2820 slow_operation_alarm.cc:140] The operation took 2m17.742922326s

********************************
[Compiling module jit_while for GPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************


# Results

In [ ]:
cpu_losses = jax.device_get(batch_losses)
best_idx = int(jnp.nanargmin(jnp.array(cpu_losses)))
best_nodes = pop_nodes[best_idx]
best_conns = pop_conns[best_idx]

print(f"Mejor individuo: loss = {cpu_losses[best_idx]:.6f}")

best_transformed = jax.jit(g.transform)(state, best_nodes, best_conns)
forward_fn = jax.jit(g.forward, static_argnums=(0,))

print("\nPredicciones vs targets:")
for i in range(len(material_ids)):
    x = jnp.array(X_train[i:i+1])
    pred = forward_fn(state, best_transformed, x[0])
    target = Y_train[i]

    pred_lat = jax.device_get(pred[:LATTICE_PARAMS])
    tgt_lat  = target[:LATTICE_PARAMS]

    print(f"\n{material_ids[i]}:")
    print(f"  Lattice real: {jnp.round(jnp.array(tgt_lat), 4)}")
    print(f"  Lattice pred: {jnp.round(jnp.array(pred_lat), 4)}")

Mejor individuo: loss = 3.940418

Predicciones vs targets:

mp-1550:
  Lattice real: [0.774  0.774  0.774  0.3333 0.3333 0.3333]
  Lattice pred: [0.6768 0.9246 0.4587 0.4707 0.1513 0.2627]

mp-2624:
  Lattice real: [0.87469995 0.87469995 0.87469995 0.3333     0.3333     0.3333    ]
  Lattice pred: [0.75869995 1.0761     0.562      0.38799998 0.1894     0.1654    ]

mp-1342:
  Lattice real: [0.7892 0.7892 0.7892 0.3333 0.3333 0.3333]
  Lattice pred: [ 0.71819997  0.6688      0.3615     -0.0247      0.11        0.25      ]

mp-3347313:
  Lattice real: [0.4913 0.4913 0.7391 0.4121 0.4121 0.3333]
  Lattice pred: [0.9069     0.5523     0.6879     0.3146     0.45499998 0.1174    ]

mp-2469:
  Lattice real: [0.83239996 0.83239996 0.83239996 0.3333     0.3333     0.3333    ]
  Lattice pred: [0.7012     0.6019     0.5879     0.0837     0.20099999 0.31399998]

mp-2691:
  Lattice real: [0.8684 0.8684 0.8684 0.3333 0.3333 0.3333]
  Lattice pred: [0.6884     0.6501     0.80619997 0.2324     0.2104 